# THỰC NGHIỆM MÔ HÌNH ĐỀ XUẤT: YOLOv8n TRÊN BỘ DỮ LIỆU GRAZPEDWRI-DX

**Mục tiêu:** Huấn luyện `YOLOv8n` (Anchor-free, ~11.2M tham số) trên **cùng tập dữ liệu đã chia ở Baseline `YOLOv5m` (`seed=42`: 15.327 Train / 4.000 Val / 1.000 Test)** để so sánh công bằng.

**Tối ưu trên Kaggle:**
- Chuyển ảnh về chuẩn 8-bit tối đa 800px trên bộ nhớ cục bộ `/kaggle/temp/dataset` (chạy train chỉ mất ~1.5 - 2.5 tiếng thay vì 8 tiếng).
- Sử dụng **2 GPU Tesla T4 (DDP)**.
- Tự động đánh giá tập Test và nén kết quả (`best.pt`, `results.csv`, biểu đồ) ra `/kaggle/working/yolov8n_results.zip`.

## Ô 1: Cài đặt thư viện Ultralytics (YOLOv8) & Kiểm tra 2 GPU

In [ ]:
!pip install -q ultralytics
import torch
import ultralytics

ultralytics.checks()
!yolo settings wandb=False

gpu_count = torch.cuda.device_count()
print(f"\n✅ Số GPU sẵn sàng: {gpu_count}")
for i in range(gpu_count):
    print(f"   - GPU {i}: {torch.cuda.get_device_name(i)}")
assert gpu_count >= 1, "Vui lòng bật Accelerator GPU T4 x2 trong Settings!"

## Ô 2: Chuẩn bị dữ liệu tốc độ cao vào `/kaggle/temp/dataset`
Giữ nguyên `random.seed(42)` để có tập Train/Val/Test khớp 100% với Baseline `YOLOv5m`.

In [ ]:
import os
import shutil
import random
import yaml
import cv2
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# 1. Quét đường dẫn ảnh và nhãn (bỏ qua thư mục không dùng để quét trong 1 giây)
print("1. Đang định vị ảnh và nhãn từ /kaggle/input...")
all_images = []
label_dict = {}
meta_file = None

for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
    dirs[:] = [d for d in sorted(dirs) if d not in ("pascalvoc", "supervisely", "notebooks")]
    base_name = os.path.basename(root)
    
    if "meta.yaml" in files:
        meta_file = Path(root) / "meta.yaml"
    if base_name == "labels":
        for f in files:
            if f.endswith(".txt"):
                p = Path(root) / f
                label_dict[p.stem] = p
    else:
        for f in files:
            if f.endswith(".png"):
                all_images.append(Path(root) / f)

all_images = sorted(all_images)
print(f"-> Tìm thấy: {len(all_images)} ảnh PNG và {len(label_dict)} nhãn TXT.")

# Đọc thông tin class từ meta.yaml gốc
orig_meta = {}
if meta_file and meta_file.exists():
    with open(meta_file, 'r') as f:
        orig_meta = yaml.safe_load(f) or {}

# 2. Chia tập giống hệt Baseline YOLOv5m (seed=42)
random.seed(42)
shuffled_images = all_images.copy()
random.shuffle(shuffled_images)

splits = {
    'train': shuffled_images[:15327],
    'val': shuffled_images[15327:15327 + 4000],
    'test': shuffled_images[15327 + 4000:15327 + 4000 + 1000]
}

# 3. Chuyển dữ liệu sang ổ cục bộ /kaggle/temp/dataset để GPU đọc siêu nhanh
base_dir = Path("/kaggle/temp/dataset")
if base_dir.exists():
    shutil.rmtree(base_dir)

tasks = []
for s_name, img_list in splits.items():
    (base_dir / "images" / s_name).mkdir(parents=True, exist_ok=True)
    (base_dir / "labels" / s_name).mkdir(parents=True, exist_ok=True)
    for img_p in img_list:
        img_dst = base_dir / "images" / s_name / img_p.name
        lbl_src = label_dict.get(img_p.stem, None)
        lbl_dst = base_dir / "labels" / s_name / f"{img_p.stem}.txt"
        tasks.append((img_p, img_dst, lbl_src, lbl_dst))

def process_one(task):
    img_src, img_dst, lbl_src, lbl_dst = task
    im = cv2.imread(str(img_src), cv2.IMREAD_GRAYSCALE)
    if im is not None:
        h, w = im.shape[:2]
        if max(h, w) > 800:
            scale = 800.0 / max(h, w)
            im = cv2.resize(im, (int(round(w * scale)), int(round(h * scale))), interpolation=cv2.INTER_AREA)
        cv2.imwrite(str(img_dst), im)
    else:
        shutil.copy2(img_src, img_dst)
    if lbl_src is not None:
        shutil.copy2(lbl_src, lbl_dst)

print(f"2. Đang tối ưu và chuyển {len(tasks)} ảnh sang /kaggle/temp/dataset (mất vài phút, chỉ chạy 1 lần)...")
with ThreadPoolExecutor(max_workers=8) as ex:
    list(tqdm(ex.map(process_one, tasks), total=len(tasks), mininterval=5.0))

# 4. Tạo file cấu hình YAML cho YOLOv8
names_list = orig_meta.get('names', [
    "boneanomaly", "bonelesion", "foreignbody", "fracture", 
    "metal", "periostealreaction", "pronatorsign", "softtissue", "text"
])
data_yaml = {
    'path': str(base_dir),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': orig_meta.get('nc', 9),
    'names': names_list
}

yaml_path = Path("/kaggle/working/grazpedwri_v8.yaml")
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(f"\n✅ HOÀN TẤT CHUẨN BỊ DỮ LIỆU! Train={len(splits['train'])}, Val={len(splits['val'])}, Test={len(splits['test'])}")

## Ô 3: Huấn luyện YOLOv8n (2 GPU) -> Đánh giá tập Test -> Nén Output

In [ ]:
%cd /kaggle/working

# 1. Huấn luyện YOLOv8n trên 2 GPU Tesla T4 (50 epochs, imgsz=640, batch=32)
!yolo detect train \
    data=/kaggle/working/grazpedwri_v8.yaml \
    model=yolov8n.pt \
    epochs=50 \
    imgsz=640 \
    batch=32 \
    device=0,1 \
    workers=4 \
    exist_ok=True \
    project=/kaggle/working/runs_v8 \
    name=yolov8n_grazpedwri

# 2. Tự động chạy đánh giá chính thức trên tập TEST (1.000 ảnh)
print("\n======================================================")
print("ĐANG ĐÁNH GIÁ MÔ HÌNH YOLOv8n TỐT NHẤT TRÊN TẬP TEST...")
print("======================================================")
!yolo detect val \
    data=/kaggle/working/grazpedwri_v8.yaml \
    model=/kaggle/working/runs_v8/yolov8n_grazpedwri/weights/best.pt \
    split=test \
    imgsz=640 \
    device=0 \
    exist_ok=True \
    project=/kaggle/working/runs_v8 \
    name=test_eval

# 3. Nén toàn bộ trọng số (best.pt), bảng kết quả (results.csv) và biểu đồ thành 1 file zip
!zip -r -q /kaggle/working/yolov8n_results.zip /kaggle/working/runs_v8
print("\n✅ ĐÃ HOÀN TẤT TOÀN BỘ VÀ NÉN FILE /kaggle/working/yolov8n_results.zip!")